# Dyslev AI — Retraining EfficientNet-B0 (Calibrated + Balanced) di Google Colab

Notebook ini melatih ulang model **EfficientNet-B0** untuk deteksi indikasi disleksia/disgrafia dengan perbaikan utama:
- **Label Smoothing (`0.1`) & Dropout (`0.3`)**: mencegah model *overconfident* (99–100%) saat prediksi salah.
- **Balanced Class Weights**: menyeimbangkan bobot kelas `Disleksia` & `Normal` untuk menekan *False Negative*.
- **Callbacks (`ModelCheckpoint`, `EarlyStopping`, `ReduceLROnPlateau`)**: otomatis menyimpan bobot terbaik dan mencegah *overfitting*.
- **Konversi Otomatis ke TFLite (`dyslexia_model_v2.tflite`)**.

> **Pastikan Runtime Google Colab sudah menggunakan GPU:** `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
# 1. Cek GPU & Clone Repository GitHub Dyslev-AI
!nvidia-smi
!git clone https://github.com/danishalzam8002belajarid-learn/Dyslev-AI-.git
%cd /content/Dyslev-AI-

In [ ]:
# 2. Hubungkan Google Drive & Siapkan Dataset
# Sebelumnya, kompres folder 'data/' di laptop Anda menjadi 'data.zip' dan upload ke Google Drive (MyDrive)
from google.colab import drive
import os

drive.mount('/content/drive')

# Ganti path di bawah jika Anda menaruh data.zip di subfolder tertentu di Google Drive
DRIVE_ZIP_PATH = '/content/drive/MyDrive/data.zip'

if os.path.exists(DRIVE_ZIP_PATH):
    !unzip -q "{DRIVE_ZIP_PATH}" -d /content/Dyslev-AI-/
    print('Dataset berhasil diekstrak!')
else:
    print(f'File {DRIVE_ZIP_PATH} tidak ditemukan. Pastikan data.zip sudah diupload ke Google Drive!')

# Verifikasi struktur folder data/train
for cls in ['Disleksia', 'Normal']:
    p = os.path.join('/content/Dyslev-AI-/data/train', cls)
    cnt = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f'Jumlah gambar {cls}: {cnt}')

In [ ]:
# 3. Jalankan Pipeline Training & Konversi TFLite
# Skrip train_and_convert.py sudah dilengkapi Label Smoothing, Class Weight, & EarlyStopping
!python train_and_convert.py

In [ ]:
# 4. Evaluasi Singkat Hasil Model Baru (TFLite) pada Sampel Validasi
import glob, random, math
import numpy as np
from PIL import Image
import tensorflow as tf

interp = tf.lite.Interpreter(model_path='dyslexia_model_v2.tflite', num_threads=4)
interp.allocate_tensors()
inp_idx = interp.get_input_details()[0]['index']
out_idx = interp.get_output_details()[0]['index']

dys_files = glob.glob('data/train/Disleksia/*')
norm_files = glob.glob('data/train/Normal/*')
random.seed(123)
test_samples = [(p, 1) for p in random.sample(dys_files, min(250, len(dys_files)))] + \
               [(p, 0) for p in random.sample(norm_files, min(250, len(norm_files)))]

tp = tn = fp = fn = 0
thresh = 0.50
for p, y in test_samples:
    im = Image.open(p).convert('RGB').resize((224, 224))
    arr = np.expand_dims(np.asarray(im, dtype=np.float32) / 255.0, axis=0)
    interp.set_tensor(inp_idx, arr)
    interp.invoke()
    prob = float(interp.get_tensor(out_idx)[0][0])
    pred = 1 if prob > thresh else 0
    if y == 1 and pred == 1: tp += 1
    elif y == 0 and pred == 0: tn += 1
    elif y == 0 and pred == 1: fp += 1
    else: fn += 1

total = tp + tn + fp + fn
print(f'Evaluasi 500 Sampel (threshold={thresh}):')
print(f'TP={tp}, TN={tn}, FP={fp}, FN={fn}')
print(f'Accuracy: {(tp+tn)/total*100:.2f}% | Recall (Sensitivitas): {tp/max(tp+fn,1)*100:.2f}% | Spesifisitas: {tn/max(tn+fp,1)*100:.2f}%')

In [ ]:
# 5. Simpan Model ke Google Drive & Download ke Laptop
import shutil
from google.colab import files

OUT_DRIVE_DIR = '/content/drive/MyDrive/Dyslev_AI_Models'
os.makedirs(OUT_DRIVE_DIR, exist_ok=True)

for fname in ['dyslexia_model.h5', 'dyslexia_model_v2.tflite']:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(OUT_DRIVE_DIR, fname))
        print(f'Tersimpan di Google Drive: {OUT_DRIVE_DIR}/{fname}')
        files.download(fname)